🧾 PROJECT PLAN – PUBMED + TRANSFORMER

PHASE 1 — PLANNING
	1.	Define NLP task (e.g., publication type classification)
	2.	Define label set (e.g., Review, Clinical Trial, Case Report, Meta-Analysis, Others)
	3.	Choose API (NCBI Entrez)
	4.	Choose transformer models (e.g., SciBERT + BioBERT + BERT-base)
	5.	Choose metrics (accuracy + F1 + confusion matrix)

⸻

PHASE 2 — DATA COLLECTION
	6.	Query PubMed API for papers
	7.	Download PMIDs for target categories
	8.	Fetch metadata via efetch
	9.	Extract:
	•	title
	•	abstract
	•	publication type
	•	year
	10.	Store raw JSON/XML
	11.	Convert to DataFrame
	12.	Save as raw CSV/JSON

⸻

PHASE 3 — DATA PREPARATION
	13.	Filter papers with missing abstracts
	14.	Filter papers without target labels
	15.	Map labels to numeric IDs
	16.	Remove duplicates
	17.	Remove very short abstracts
	18.	Split into train/val/test sets
	19.	Save cleaned dataset

⸻

PHASE 4 — TEXT PROCESSING
	20.	Tokenize text with transformer tokenizer
	21.	Truncate or pad to max sequence length
	22.	Convert to HuggingFace Dataset format

⸻

PHASE 5 — MODEL TRAINING
	23.	Load transformer model (e.g., SciBERT)
	24.	Configure training arguments
	25.	Fine-tune on train set
	26.	Validate on val set each epoch
	27.	Save best-performing checkpoint

⸻

PHASE 6 — EVALUATION
	28.	Load test set
	29.	Generate predictions
	30.	Compute:

	•	accuracy
	•	precision
	•	recall
	•	F1

	31.	Compute confusion matrix
	32.	Perform error analysis (inspect misclassified samples)

⸻

PHASE 7 — COMPARISON (OPTIONAL BUT STRONG)
	33.	Train baseline transformer (e.g., BERT-base)
	34.	Train domain transformer (e.g., SciBERT/BioBERT)
	35.	Compare performance on test set
	36.	Compare inference time / model size

⸻

PHASE 8 — DOCUMENTATION + BUSINESS VALUE
	37.	Explain medical classification relevance
	38.	Explain independent data collection
	39.	Document API usage
	40.	Document preprocessing choices
	41.	Document transformer architecture briefly
	42.	Document evaluation results
	43.	State limitations
	44.	State possible improvements
	45.	State business/scientific use cases

⸻

PHASE 9 — FINAL ARTIFACTS
	46.	Final cleaned dataset (CSV)
	47.	Training notebook
	48.	Evaluation notebook
	49.	Model result plots
	50.	Report / presentation slides

In [ ]:
#pip install requests pandas

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached pandas-2.3.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached charset_normalizer-3.4.4-cp313-cp313-macosx_10_13_universal2.whl.metadata (37 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
Using cached charset_normalizer-3.4.4-cp313-cp313-macosx_10_13_universal2.whl (208 kB)
Using cached idna-3.11-py3-none-any.whl (71 kB)
Using cached pandas-2.3.3-cp313-cp313-macosx_11_0_arm64.whl (10.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 3.5 MB/s  0:00:01 eta 0:00:01
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [pandas]2m8/9 [pandas]
Note: you may need to restart the kern

In [ ]:
import requests
import time
import xml.etree.ElementTree as ET
import pandas as pd
from pathlib import Path

In [ ]:
# CONFIG
EMAIL = "w.madro@student.edu.com"  
RETMAX = 200                   
SAVE_DIR = Path("pubmed_raw")
SAVE_DIR.mkdir(exist_ok=True)

In [ ]:
# Search terms for specific publication types
SEARCH_QUERIES = {
    "clinical_trial": "clinical trial[pt]",
    "review": "review[pt]",
    "case_report": "case reports[pt]",
    "letter": "letter[pt]",
    "meta_analysis": "meta-analysis[pt]"
}

In [ ]:
# 6–7. Query PubMed API & download PMIDs
# -----------------------------
def search_pubmed(term, retmax=RETMAX):
    """
    Return a list of PMIDs for the given PubMed query.
    """
    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {
        "db": "pubmed",
        "term": term,
        "retmax": retmax,
        "retmode": "json",
        "email": EMAIL
    }
    r = requests.get(base, params=params)
    r.raise_for_status()
    data = r.json()
    pmids = data["esearchresult"]["idlist"]
    return pmids


# -----------------------------
# 8–9. Fetch metadata via efetch & extract fields
# -----------------------------
def fetch_details(pmids, raw_xml_path=None, sleep_sec=0.4):
    """
    Fetch PubMed details for a list of PMIDs and extract:
    title, abstract, publication types, year.
    Optionally save raw XML.
    """
    if not pmids:
        return []

    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    ids_str = ",".join(pmids)
    params = {
        "db": "pubmed",
        "id": ids_str,
        "retmode": "xml",
        "email": EMAIL
    }

    r = requests.get(base, params=params)
    r.raise_for_status()
    xml_text = r.text

    # 10. Store raw XML if requested
    if raw_xml_path is not None:
        raw_xml_path.write_text(xml_text, encoding="utf-8")

    # Be polite
    time.sleep(sleep_sec)

    # Parse XML and extract fields
    root = ET.fromstring(xml_text)
    records = []

    for article in root.findall(".//PubmedArticle"):
        # PMID
        pmid_el = article.find(".//MedlineCitation/PMID")
        pmid = pmid_el.text if pmid_el is not None else None

        # Title
        title_el = article.find(".//MedlineCitation/Article/ArticleTitle")
        title = ET.tostring(title_el, encoding="unicode", method="text").strip() if title_el is not None else None

        # Abstract (may be multiple AbstractText elements)
        abstract_elems = article.findall(".//MedlineCitation/Article/Abstract/AbstractText")
        if abstract_elems:
            abstract_texts = [ET.tostring(a, encoding="unicode", method="text").strip() for a in abstract_elems]
            abstract = "\n".join(abstract_texts)
        else:
            abstract = None

        # Publication types (list)
        pt_elems = article.findall(".//MedlineCitation/Article/PublicationTypeList/PublicationType")
        pub_types = [pt.text for pt in pt_elems if pt is not None and pt.text] if pt_elems else []

        # Year (from PubDate; sometimes stored differently)
        year = None
        year_el = article.find(".//MedlineCitation/Article/Journal/JournalIssue/PubDate/Year")
        if year_el is not None and year_el.text:
            year = year_el.text
        else:
            # Try MedlineDate (e.g. "2005 Jan-Feb")
            md_el = article.find(".//MedlineCitation/Article/Journal/JournalIssue/PubDate/MedlineDate")
            if md_el is not None and md_el.text:
                year = md_el.text[:4]  # first 4 chars are usually year

        records.append(
            {
                "pmid": pmid,
                "title": title,
                "abstract": abstract,
                "pub_types": pub_types,
                "year": year,
            }
        )

    return records


# -----------------------------
# MAIN: Run full pipeline 6–12
# -----------------------------
all_records = []

for label_name, query in SEARCH_QUERIES.items():
    print(f"Querying PubMed for: {label_name} → {query}")

    # 6–7: get PMIDs
    pmids = search_pubmed(query, retmax=RETMAX)
    print(f"  Found {len(pmids)} PMIDs")

    if not pmids:
        continue

    # 8–10: fetch details + save raw XML per label
    raw_xml_file = SAVE_DIR / f"pubmed_{label_name}.xml"
    records = fetch_details(pmids, raw_xml_path=raw_xml_file)

    # Add a label column for classification later
    for rec in records:
        rec["label_name"] = label_name

    all_records.extend(records)

print(f"Total records collected: {len(all_records)}")

# 11. Convert to DataFrame
df = pd.DataFrame(all_records)

# Optional: basic cleaning (drop rows without abstract)
df = df.dropna(subset=["abstract"]).reset_index(drop=True)

# 12. Save as raw CSV and JSON
df.to_csv("pubmed_raw_dataset.csv", index=False)
df.to_json("pubmed_raw_dataset.json", orient="records", force_ascii=False)

print("Saved pubmed_raw_dataset.csv and pubmed_raw_dataset.json")

Querying PubMed for: clinical_trial → clinical trial[pt]
  Found 200 PMIDs
Querying PubMed for: review → review[pt]
  Found 200 PMIDs
Querying PubMed for: case_report → case reports[pt]
  Found 200 PMIDs
Total records collected: 600
Saved pubmed_raw_dataset.csv and pubmed_raw_dataset.json
